<!-- nav -->
[← 6 · Drawing values](06_Drawing_Values.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 7 · Generating test data

**The problem.** Tests need data that looks like production's mix, reproducibly: the same seed must give the same
contacts tomorrow, in CI, and in the TypeScript test suite. `Generators.Generate` builds new matches of a pattern.

In [1]:
from collections import Counter
from statistics import mean

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Distributions, Generators, Predicates, Queries, Validators
from mbse.Schemas.Framework import Stores
from toolkit import Contact, Directory, book, listed

person = E.variable("person")
HasName = (Predicates.OfPredicate.Builder().name("HasName").symbols({"person": Contact})
           .parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())


def aged(distribution):
    return distribution.symbol("age").requires(lambda person, age: person.age == age).create()


People = (Predicates.OfPredicate.Builder().name("People").symbols({"person": Contact}).requires(
    Distributions.Choices.Builder().arms(
        lambda a: a.weight(3).requires(HasName(person, "senior"), person.age.ge(65),
                                       aged(Distributions.Normal.Builder().mean(70).deviation(8).rounded())),
        lambda a: a.weight(7).requires(HasName(person, "adult"), aged(Distributions.Uniform.Builder().low(18).high(64))),
    ).decreasing().create()).create())

## Step 1: generating

Each step chooses an arm by weight, draws the values the arm's distributions give, sets the properties the equalities
say, and builds a contact with the store's builders:

In [2]:
store = book()
generated = Generators.Generate(store, People, Stores.PCG32(42))
people = [next(generated)["person"] for _ in range(1000)]
[(p.name, p.age) for p in people[:6]]

[('adult', 51),
 ('senior', 66),
 ('adult', 48),
 ('adult', 27),
 ('adult', 48),
 ('adult', 24)]

In [3]:
seniors = [p.age for p in people if p.name == "senior"]
Counter(p.name for p in people), round(mean(seniors), 1), min(seniors)

(Counter({'adult': 717, 'senior': 283}), 73.6, 65)

About three in ten are seniors, their ages near 70 and never under 65. A senior's normal can draw 60; then the arm
doesn't hold, and the step draws again. That's **rejection**: the generator counts it, so you can see what a rule
costs:

In [4]:
generated.steps, generated.rejected

(1000, 103)

## Step 2: the same seed, the same data

Each step draws from its own stream, split from the seed, so the tenth contact doesn't depend on how many came before
it, and the same seed gives the same contacts. The TypeScript generator gives these exact ones too: both implement the
sampling, and even `log` and `exp`, with the same arithmetic.

In [5]:
again = Generators.Generate(book(), People, Stores.PCG32(42))
[next(again)["person"].age for _ in range(6)] == [p.age for p in people[:6]]

True

## Step 3: generated data is ordinary data

The contacts are transient until you list them. Then they validate and query like any other data:

In [6]:
listed(store, *people[:20])
problems = Validators.Validate(store, [People]).Reachable(Directory, store.singleton("book.Directory"))
seniors_found = [m["person"].name for m in Queries.Scan(store).select(People) if m["person"].name == "senior"]
problems, len(seniors_found)

([], 4)

## Step 4: what a generator can't do

A generator sets what equalities say and draws what distributions give. It doesn't invent links: a pattern that
requires a contact to have a phone can't be built from nothing, and says so:

In [7]:
from toolkit import Phone

p = E.variable("p")
Phoned = (Predicates.OfPredicate.Builder().symbols({"person": Contact}).requires(Predicates.Exists(
    lambda q: q.symbols({"p": Phone}).requires(Predicates.Contains(person.phones, lambda e: e.phone == p)))).create())
try:
    next(Generators.Generate(book(), Phoned, Stores.PCG32(1)))
except ValueError as error:
    print(error)

the predicate cannot be generated from its equalities and choices: what they build does not satisfy it


## Why it works this way

- **Generate from the rule you check with.** The pattern is a predicate, so what you generate passes your validators by
  construction: the generator checks every match before it yields it.
- **Rejection is honest.** Truncating a normal by redrawing gives exactly the normal restricted to the arm, and the
  count of rejections tells you when a rule makes generation expensive.
- **Reproducible across languages.** A seed is a contract: the same contacts in Python and TypeScript, today and
  later.

<!-- nav -->
[← 6 · Drawing values](06_Drawing_Values.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)